# TUTORIAL: ¿CÓMO AFINAR LOS HIPER-PARÁMETROS DE UNA RED NEURONAL CON KERAS TUNER?

En este tutorial veremos un tutorial básico sobre cómo usar Keras Tuner para afinar los hiperparámetros de una Red Neuronal.

Contenido:

1. [¿Qué son los hiperparámetros?](#scrollTo=A3bp1YrpGQAI&line=13&uniqifier=1)
2. [¿Qué es el ajuste o afinación de hiperparámetros?](#scrollTo=3jGHchIPJOrL&line=9&uniqifier=1)
3. [El problema a resolver](#scrollTo=dljQnqglJtgV&line=5&uniqifier=1)
4. [El set de datos](#scrollTo=dEgFSAxkLce4&line=24&uniqifier=1)
5. [Pre-procesamiento del set de datos](#scrollTo=4gUZsJjdNa5i&line=11&uniqifier=1)
6. [🔥🔥🔥Uso de Keras Tuner🔥🔥🔥](#scrollTo=czFmIO7pU337&line=1&uniqifier=1)
7. [Generar predicciones con el modelo afinado](#scrollTo=BgXGkY69BTfZ&line=1&uniqifier=1)

## 1. ¿Qué son los hiperparámetros?


> Los hiperparámetros son un **conjunto de variables numéricas** que nosotros, como diseñadores del modelo, **debemos escoger** correctamente para lograr obtener las mejores predicciones posibles.

Por ejemplo, en una Red Neuronal los hiperparámetros serían:

![](https://drive.google.com/uc?export=view&id=1rxHLiTwXVMdjNhdC70oSraWGwXwN36sb)

- El número de capas ocultas
- El número de neuronas en cada capa oculta
- La tasa de aprendizaje del optimizador

## 2. ¿Qué es el ajuste o afinación de hiperparámetros?

> El ajuste de hiperparámetros consiste en **encontrar la mejor combinación de hiperparámetros posible que maximice el desempeño del modelo**

En nuestro caso la idea es encontrar los mejores valores para el número de capas ocultas, el número de neuronas por capa y la tasa de aprendizaje del optimizador que permitan que la Red Neuronal genere las mejores predicciones.

Keras Tuner permite afinar estos hiperparámetros.

## 3. El problema a resolver

Para ver cómo afinar los hiperparámetros usando Keras Tuner entrenaremos una sencilla Red Neuronal que tomará como entrada datos demográficos (edad, tipo de trabajo, nivel de educación, estado civil, etc.) y que aprenderá a predecir si el nivel de ingresos de la persona será menor o igual a 50.000 USD (0) o mayor a 50.000 USD (1):

![](https://drive.google.com/uc?export=view&id=1ryAI3C3nv4Y7Rj0uIdcaEvIh3o0yLYtP)

## 4. El set de datos

El set de datos se puede descargar desde [este enlace](https://drive.google.com/file/d/1g0JqE_qKh2qa_gSQqGAr_BHK6uduCLLN/view?usp=sharing).

El set de datos contiene un total de 48.842 registros de personas adultas en Estados Unidos.

Por cada persona se han recolectado estas variables:

1. `edad`
2. `tipo_trabajo`
3. `estim_pob` (población estimada con el mismo perfil de la persona)
4. `nivel_educ`
5. `estado_civil`
6. `ocupación`
7. `relación` (parentesco que describe a la persona)
8. 	`incr_capital` (nivel de incremento del patrimonio en el año anterior)
9. `perd_capital` (nivel de reducción del patrimonio en el año anterior)
10.	`horas_semana` (número de horas laboradas por semana)
11. `nacionalidad``
12. `ingresos` (menores o iguales a 50.000 USD - 0 - o mayores a 50.000 USD)

Las entradas al modelo serán las variables 1 a 11 y el modelo deberá aprender a predecir la variable 12 (ingresos).

Comencemos leyendo el set de datos. Después de descargarlo del enlace lo llevamos al disco duro de la máquina virtual de Google Colab y de allí lo leemos con Pandas:

In [2]:
# Leer dataset
import pandas as pd

df = pd.read_csv('/content/demografia_adultos.csv')
df

,edad,tipo_trabajo,estim_pob,nivel_educ,estado_civil,ocupación,relación,incr_capital,perd_capital,horas_semana,nacionalidad,ingresos
0,25,privado,226802,7,soltero(a),operario,hijo_único,0,0,40,Estados_Unidos,<=50K
1,38,privado,89814,9,casado(a),agricultura_pesca,esposo,0,0,50,Estados_Unidos,<=50K
2,28,gob_local,336951,12,casado(a),seguridad,esposo,0,0,40,Estados_Unidos,>50K
3,44,privado,160323,10,casado(a),operario,esposo,7688,0,40,Estados_Unidos,>50K
4,18,privado,103497,10,soltero(a),especialidad_profesional,hijo_único,0,0,30,Estados_Unidos,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...
48837,27,privado,257302,12,casado(a),soporte_técnico,esposa,0,0,38,Estados_Unidos,<=50K
48838,40,privado,154374,9,casado(a),operario,esposo,0,0,40,Estados_Unidos,>50K
48839,58,privado,151910,9,soltero(a),administrativo,soltero,0,0,40,Estados_Unidos,<=50K
48840,22,privado,201490,9,soltero(a),administrativo,hijo_único,0,0,20,Estados_Unidos,<=50K


Hagamos una breve exploración del set de datos.

Comencemos viendo las características de las variables numéricas:

In [3]:
# Exploración variales numéricas
df.describe()

,edad,estim_pob,nivel_educ,incr_capital,perd_capital,horas_semana
count,48842.000000,4.884200e+04,48842.000000,48842.000000,48842.000000,48842.000000
mean,38.643585,1.896641e+05,10.078089,1079.067626,87.502314,40.422382
std,13.710510,1.056040e+05,2.570973,7452.019058,403.004552,12.391444
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.175505e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.781445e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.376420e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.490400e+06,16.000000,99999.000000,4356.000000,99.000000


Vemos que las variables numéricas (edad, estim_pob, nivel_educ, etc.) tienen rangos de valores diferentes. Antes de llevarlas a la Red Neuronal tendremos que escalarlas.

Hagamos ahora la exploración de las variables categóricas (almacenadas como tipo `object` en el *DataFrame*):

In [4]:
# Exploración variables categóricas

# Listados para almacenar los nombres de las variables numéricas y categóricas
col_categoricas = []
col_numericas = []

# Iterar por las columnas del DataFrame
for columna in df.columns:

    # Si la columna es categórica
    if df[columna].dtype == 'object':
        # Imprimir los sub-niveles de la variable categórica
        print(f'{columna}: {df[columna].unique()}')

        # Y almacenar el nombre de la columna en el listado
        col_categoricas.append(columna)
    else:
        # Almacenar el nombre de la columna numérica en el listado
        col_numericas.append(columna)

tipo_trabajo: ['privado' 'gob_local' 'autónomo' 'gob_federal' 'gob_estatal' 'empresario'
 'desempleado' 'no_ha_trabajado']
estado_civil: ['soltero(a)' 'casado(a)' 'divorciado(a)']
ocupación: ['operario' 'agricultura_pesca' 'seguridad' 'especialidad_profesional'
 'otros' 'mantenimiento' 'administrativo' 'ejecutivo' 'soporte_técnico'
 'ventas' 'reparación' 'transporte_mudanza' 'limpieza' 'fuerzas_armadas']
relación: ['hijo_único' 'esposo' 'sin_familia' 'soltero' 'esposa' 'otro']
nacionalidad: ['Estados_Unidos' 'Perú' 'Guatemala' 'México' 'Rep_Dominicana' 'Irlanda'
 'Alemania' 'Filipinas' 'Tailandia' 'Haití' 'El_Salvador' 'Puerto_Rico'
 'Vietnam' 'Corea_Del_sur' 'Colombia' 'Japón' 'India' 'Camboya' 'Polonia'
 'Laos' 'Inglaterra' 'Cuba' 'Taiwán' 'Italia' 'Canadá' 'Portugal' 'China'
 'Nicaragua' 'Honduras' 'Irán' 'Escocia' 'Jamaica' 'Ecuador' 'Croacia'
 'Hungría' 'Hong_Kong' 'Grecia' 'Trinidad_Tobago' 'Guam' 'Francia'
 'Países_bajos']
ingresos: ['<=50K' '>50K']


## 5. Pre-procesamiento del set de datos

Para poder llevar los datos a la Red Neuronal tenemos que llevar a cabo varias etapas de pre-procesamiento. La idea es representar las variables en el formato numérico más adecuado.

Estas serán las etapas a implementar:

- Codificar las columnas categóricas en el formato one-hot (exceptuando la columna "ingresos")
- Codificar la columna categórica "ingresos" en formato binario (0: ingresos menores o iguales a 50.000, 1: ingresos mayores a 50.000)
- Crear los sets de entrenamiento, validación y prueba
- Escalar las variables numéricas para que estén en el rango de 0 a 1
- Conformar los arreglos NumPy que usaremos a la entrada y salida de la Red Neuronal

Veamos cada una de estas fases.

### 5.1. Codificación *one-hot* de las variables categóricas

La idea es representar cada variable categórica como una secuencia de 0s y 1s.

Por ejemplo, para la variable `estado_civil` la codificación *one-hot* sería algo similar a esto:

`soltero(a)` → [1, 0, 0]
`casado(a)` → [0, 1, 0]
`divorciado(a)` → [0, 0, 1]

Esto se puede lograr fácilmente en una sola línea de código usando el método `get_dummies` de pandas y teniendo en cuenta que la codificación la haremos para todas las variables categóricas **exceptuando "ingresos"**:

In [5]:
col_categoricas

['tipo_trabajo',
 'estado_civil',
 'ocupación',
 'relación',
 'nacionalidad',
 'ingresos']

In [6]:
# Codificación one-hot de las columnas categóricas exceptuando "ingresos"
df2 = pd.get_dummies(df,columns=col_categoricas[:-1])
df2.shape

(48842, 79)

### 5.2. Codificación de la columna "ingresos" en formato binario

En este caso la idea es tomar esta columna y codificar sus dos niveles de la siguiente forma:

- `<=50K` → 0
- `>50K` → 1

Esto también lo podemos hacer de forma muy sencilla usando `get_dummies()` pero agregando el argumento `drop_first=True` para que en lugar de codificar en formato *one-hot* ([1,0] y [0,1]) la codificación sea binaria:

In [7]:
# Codificar 0/1 columna ingresos
df2 = pd.get_dummies(df2, columns=['ingresos'], drop_first=True)
df2

,edad,estim_pob,nivel_educ,incr_capital,perd_capital,horas_semana,tipo_trabajo_autónomo,tipo_trabajo_desempleado,tipo_trabajo_empresario,tipo_trabajo_gob_estatal,...,nacionalidad_Perú,nacionalidad_Polonia,nacionalidad_Portugal,nacionalidad_Puerto_Rico,nacionalidad_Rep_Dominicana,nacionalidad_Tailandia,nacionalidad_Taiwán,nacionalidad_Trinidad_Tobago,nacionalidad_Vietnam,ingresos_>50K
0,25,226802,7,0,0,40,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,38,89814,9,0,0,50,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,28,336951,12,0,0,40,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
3,44,160323,10,7688,0,40,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
4,18,103497,10,0,0,30,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48837,27,257302,12,0,0,38,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
48838,40,154374,9,0,0,40,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
48839,58,151910,9,0,0,40,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
48840,22,201490,9,0,0,20,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [ ]:
df2.shape

(48842, 79)

### 5.3. Crear los sets de entrenamiento, validación y prueba

- El set de entrenamiento permite obtener de forma automática los parámetros del modelo
- 👉👉👉**El set de validación se usa para afinar los hiperparámetros del modelo**👈👈👈
- El set de prueba se usa para generar predicciones con el modelo entrenado

En este caso haremos la partición de la siguiente forma:

- Entrenamiento: 70% de los datos
- Validación: 20% de los datos
- Prueba: 10% restante

Comencemos mezclando aleatoriamente las filas del *DataFrame* (y fijando la semilla del generador aleatorio para que esta mezcla aleatoria sea siempre la misma cada vez que ejecutemos esta línea de código):

In [8]:
# Mezclar aleatoriamente (semilla = 123)
df_s = df2.sample(frac=1, random_state=123)
df_s

,edad,estim_pob,nivel_educ,incr_capital,perd_capital,horas_semana,tipo_trabajo_autónomo,tipo_trabajo_desempleado,tipo_trabajo_empresario,tipo_trabajo_gob_estatal,...,nacionalidad_Perú,nacionalidad_Polonia,nacionalidad_Portugal,nacionalidad_Puerto_Rico,nacionalidad_Rep_Dominicana,nacionalidad_Tailandia,nacionalidad_Taiwán,nacionalidad_Trinidad_Tobago,nacionalidad_Vietnam,ingresos_>50K
20668,52,117700,9,0,0,40,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1722,19,351757,6,0,0,30,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
39609,31,101345,9,0,0,40,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
15858,25,324854,13,0,0,40,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
41078,36,245521,4,0,0,35,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7763,42,37997,13,0,0,40,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
15377,65,326936,9,0,0,40,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
17730,44,229466,10,0,0,50,False,False,True,False,...,False,False,False,False,False,False,False,False,False,True
28030,35,265954,13,0,0,40,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


Ahora definamos el número de datos que tendrá cada sub-set con base en el tamaño del set de datos original:

In [9]:
# Número de datos de los sets de entrenamiento, validación y prueba
N = df_s.shape[0] # Cantidad total de datos
NTRAIN = int(0.7*N)
NVAL = int(0.2*N)
NTEST = N - NTRAIN - NVAL

Y ahora sí seleccionamos los sets de entrenamiento, validación y prueba a partir del *DataFrame* mezclado:

In [10]:
# Seleccionar las filas correspondientes
df_train = df_s.iloc[0:NTRAIN,:]
df_val = df_s.iloc[NTRAIN:NTRAIN+NVAL,:]
df_test = df_s.iloc[NTRAIN+NVAL:,:]

# Imprimir información en pantalla
print(f'Tamaño set original: {df_s.shape}')
print(f'Tamaño set de entrenamiento: {df_train.shape}')
print(f'Tamaño set de validación: {df_val.shape}')
print(f'Tamaño set de prueba: {df_test.shape}')

Tamaño set original: (48842, 79)
Tamaño set de entrenamiento: (34189, 79)
Tamaño set de validación: (9768, 79)
Tamaño set de prueba: (4885, 79)


Perfecto, ya tenemos 3 dataframes correspondientes a los sets de entrenamiento, validación y prueba.

### 5.4. Escalar los datos numéricos

El siguiente paso es escalar los datos numéricos.

Como hemos usado la codificación *one-hot* para las variables categóricas, estas tendrán valores entre 0 y 1.

Así que resulta lógico codificar los valores numéricos en el rango de 0 a 1 para lo cual usaremos `MinMaxScaler` de Scikit-Learn (de la cual hablo en detalle en un tutorial anterior).

Comencemos recordando cuáles son las variables numéricas:

In [11]:
df.describe()

,edad,estim_pob,nivel_educ,incr_capital,perd_capital,horas_semana
count,48842.000000,4.884200e+04,48842.000000,48842.000000,48842.000000,48842.000000
mean,38.643585,1.896641e+05,10.078089,1079.067626,87.502314,40.422382
std,13.710510,1.056040e+05,2.570973,7452.019058,403.004552,12.391444
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.175505e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.781445e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.376420e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.490400e+06,16.000000,99999.000000,4356.000000,99.000000


Como cada variable numérica es diferente tendremos que usar un escalador para cada una de ellas.

Además, la lógica de escalamiento será:

- Se escala la variable de entrenamiento con `fit_transform()` y el resultado se almacena en el escalador correspondiente
- Se usa el resultado anterior junto con el método `transform()` para escalar la misma variable pero en los sets de validación y prueba


In [12]:
# No imprimir mensaje de advertencia
pd.options.mode.chained_assignment = None

# Importar MinMaxScaler
from sklearn.preprocessing import MinMaxScaler

# Diccionario que contendrá los escaladores para cada variable numérica
scalers = {}

# Iterar por las variables numéricas (listado "col_numericas") y por cada variable:
# 1. Crear instancia del escalador y almacenarla en un diccionario
# 2. "fit_transform" sobre el set de entrenamiento
# 3. "transform" sobre los sets de validación y prueba
for col in col_numericas:
    # 1. Instancia del escalador
    scalers[col] = MinMaxScaler()

    # 2. Fit-transform entrenamiento y reemplazar
    df_train.loc[:,col] = scalers[col].fit_transform(df_train[col].to_numpy().reshape(-1,1))

    # 3. Transform para escalar validación y prueba
    df_val.loc[:,col] = scalers[col].transform(df_val[col].to_numpy().reshape(-1,1))
    df_test.loc[:,col] = scalers[col].transform(df_test[col].to_numpy().reshape(-1,1))

# Verificar
print(f'Entrenamiento min/max: {df_train.min().min()}/{df_train.max().max()}')
print(f'Validación min/max: {df_val.min().min()}/{df_val.max().max()}')
print(f'Prueba min/max: {df_test.min().min()}/{df_test.max().max()}')

Entrenamiento min/max: 0.0/1.0
Validación min/max: 0.0/1.0
Prueba min/max: 0.0/1.0


<ipython-input-12-3e74637b31c7>:19: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.47945205 0.02739726 0.19178082 ... 0.38356164 0.42465753 0.65753425]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_train.loc[:,col] = scalers[col].fit_transform(df_train[col].to_numpy().reshape(-1,1))
<ipython-input-12-3e74637b31c7>:22: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.24657534 0.65753425 0.10958904 ... 0.30136986 0.23287671 0.49315068]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_val.loc[:,col] = scalers[col].transform(df_val[col].to_numpy().reshape(-1,1))
<ipython-input-12-3e74637b31c7>:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.24657534 0.08219178 0.04109589 ... 0.3698

### 5.5. Conformar arreglos de entrada y de salida a la Red Neuronal

Hasta ahora tenemos los sets de entrenamiento, validación y prueba en formato *DataFrame* de Pandas.

Pero para llevarlos a la Red Neuronal de Keras tendremos que convertirlos al formato NumPy.

En primer lugar observemos los tipos de dato que tenemos en los *DataFrames*:

In [13]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 34189 entries, 20668 to 42319
Data columns (total 79 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   edad                                34189 non-null  float64
 1   estim_pob                           34189 non-null  float64
 2   nivel_educ                          34189 non-null  float64
 3   incr_capital                        34189 non-null  float64
 4   perd_capital                        34189 non-null  float64
 5   horas_semana                        34189 non-null  float64
 6   tipo_trabajo_autónomo               34189 non-null  bool   
 7   tipo_trabajo_desempleado            34189 non-null  bool   
 8   tipo_trabajo_empresario             34189 non-null  bool   
 9   tipo_trabajo_gob_estatal            34189 non-null  bool   
 10  tipo_trabajo_gob_federal            34189 non-null  bool   
 11  tipo_trabajo_gob_local              34189 

Todos son datos numéricos tipo `float64` (números decimales) o `uint8` (números enteros).

Sin embargo, lo mejor es representar todas las variables como datos numéricos en formato punto flotante (`float64`).

Veamos ahora el tamaño de los *DataFrames*:

In [14]:
df_val

,edad,estim_pob,nivel_educ,incr_capital,perd_capital,horas_semana,tipo_trabajo_autónomo,tipo_trabajo_desempleado,tipo_trabajo_empresario,tipo_trabajo_gob_estatal,...,nacionalidad_Perú,nacionalidad_Polonia,nacionalidad_Portugal,nacionalidad_Puerto_Rico,nacionalidad_Rep_Dominicana,nacionalidad_Tailandia,nacionalidad_Taiwán,nacionalidad_Trinidad_Tobago,nacionalidad_Vietnam,ingresos_>50K
5087,0.246575,0.015876,0.066667,0.00000,0.0,0.142857,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
29189,0.657534,0.194308,0.400000,0.01797,0.0,0.397959,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
37656,0.109589,0.199323,0.800000,0.00000,0.0,0.448980,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
46839,0.602740,0.147119,0.066667,0.03942,0.0,0.193878,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
44637,0.328767,0.209415,0.533333,0.00000,0.0,0.071429,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23856,0.342466,0.062035,0.666667,0.00000,0.0,0.500000,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
14713,0.136986,0.075443,0.800000,0.00000,0.0,0.418367,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
19606,0.301370,0.108539,0.533333,0.00000,0.0,0.397959,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
21806,0.232877,0.066529,0.800000,0.00000,0.0,0.397959,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [15]:
print(df_train.shape)
print(df_val.shape)
print(df_test.shape)

(34189, 79)
(9768, 79)
(4885, 79)


Son arreglos de 79 columnas. Las primeras 78 serán las variables de entrada al modelo (las llamaremos "x") mientras que la última columna será la variable a predecir (la llamaremos "y").


Así que por cada *DataFrame* (`df_train`, `df_val` y `df_test`) haremos lo siguiente:

- Crearemos el arreglo *x* que será la entrada al modelo y que tomará las primeras 78 columnas
- Crearemos el arreglo *y* que será la salida que debe aprender a predecir el modelo y que corresponde a la última columna
- Para extraer los valores en formato NumPy usaremos el método `to_numpy` y agregaremos el argumento `dtype=np.float64` para convertir los valores al formato de punto flotante adecuado para Keras

Veamos cómo implementar estos pasos:

In [16]:
# Conformar arreglos X, Y
import numpy as np

# Train
x_train = df_train.iloc[:,:78].to_numpy(dtype=np.float64)
y_train = df_train.iloc[:,78].to_numpy(dtype=np.float64)

# Val
x_val = df_val.iloc[:,:78].to_numpy(dtype=np.float64)
y_val = df_val.iloc[:,78].to_numpy(dtype=np.float64)

# Test
x_test = df_test.iloc[:,:78].to_numpy(dtype=np.float64)
y_test = df_test.iloc[:,78].to_numpy(dtype=np.float64)

# Verificar tamaños
print(f'Entrenamiento (x,y): {x_train.shape}, {y_train.shape}')
print(f'Validación (x,y): {x_val.shape}, {y_val.shape}')
print(f'Prueba (x,y): {x_test.shape}, {y_test.shape}')

Entrenamiento (x,y): (34189, 78), (34189,)
Validación (x,y): (9768, 78), (9768,)
Prueba (x,y): (4885, 78), (4885,)


¡Perfecto, en este punto ya tenemos los datos listos para presentarlos al modelo!

## 6. 🔥🔥🔥Uso de Keras Tuner🔥🔥🔥

Recordemos que queremos afinar los hiperparámetros de la Red Neuronal.

Y estos hiperparámetros son:

1. El **número de capas ocultas**: probaremos con modelos de 1, 2 o 3 capas ocultas
2. El **número de neuronas por capa oculta**: probaremos con 20, 40 o 60 neuronas por capa
3. La **tasa de aprendizaje del optimizador**: probaremos con tasas de aprendizaje de 0.1, 0.01 y 0.001
3. Las **funciones de activación de las neuronas en cada capa**: probaremos con tanh y relu

Así que la afinación consiste en:

1. Crear un modelo con cada combinación de hiperparámetros
2. Entrenar y validar dicho modelo y almacenar los resultados
3. Repetir los pasos (1) y (2) para cada combinación
4. Escoger el modelo con el mejor desempeño

En este caso podremos tener un total de 54 diferentes combinaciones de hiperparámetros, pues cada uno puede tener 3 valores diferentes (3 diferentes números de capas ocultas, 3 diferentes números de neuronas por capa, 3 diferentes tasas de aprendizaje) y 2 para las funciones de activación.

Aunque suena complicado en realidad esta afinación resulta muy sencilla con Keras Tuner.

Para poder afinar el modelo usando Keras Tuner debemos llevar a cabo 2 pasos:

- Paso 1: crear una función que acepte como entrada un objeto tipo `HyperParameters()` y que nos permitirá fácilmente crear diferentes modelos para los diferentes hiperparámetros a afinar
2. Paso 2: crear un afinador (*tuner*) definiendo los criterios de afinación y usarlo para entrenar y escoger el mejor modelo tras esta afinación.

Veamos en detalle cada uno de estos pasos

### 6.1. Paso 1: escribir la función `crear_modelo`

Esta función:

- Toma como entrada un objeto tipo `HyperParameters()`.
- El objeto tipo `HyperParameters()` contendrá los hiperparámetros con los que construiremos el modelo.
- La función toma estos hiperparámetros y a la salida retorna el modelo con los hiperparámetros especificados

Veamos cómo implementar esta función, pero antes debemos instalar Keras Tuner:

In [17]:
!pip install keras-tuner -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 2.6 MB/s eta 0:00:00


Ahora sí implementemos la función, fijando inicialmente la semilla del generador aleatorio para la reproducibilidad del entrenamiento:

In [34]:
import tensorflow as tf

# Semilla generador aleatorio
tf.random.set_seed(23)

def crear_modelo(hp):
    # Contenedor vacío
    modelo = tf.keras.Sequential()

    # Capa de entrada: cada dato tendrá 78 elementos
    modelo.add(tf.keras.layers.Input(shape=(78,)))

    # =======
    # Hiperparámetro número de capas ocultas (1, 2 o 3)
    for i in range(hp.Int("num_layers",
                             min_value=1, # Mínimo número de capas ocultas
                             max_value=3, # Máximo número de capas ocultas
                             step=1 # Tamaño del salto (para obtener 1, 2 o 3)
                          )):
        # =======
        # Hiperparámetro número de neuronas por capa
        modelo.add(tf.keras.layers.Dense(
            units = hp.Int(f"units_{i}",
                           min_value=20, # Mínimo número de neuronas/capa
                           max_value=60, # Máximo número de neuronas/capa
                           step=20 # Tamaño del salto (para obtener 20, 40, 60)
                           ),
            activation=hp.Choice("activation", ["relu", "tanh"]),)
        )

    # Capa de salida
    modelo.add(tf.keras.layers.Dense(units=1, activation="sigmoid"))

    # =======
    # Hiperparámetro tasa de aprendizaje del optimizador
    hp_lr = hp.Choice("learning_rate", values=[0.1, 0.01, 0.001])

    # Compilar (optimizador, pérdida, métrica de desempeño)
    modelo.compile(
        optimizer = tf.keras.optimizers.SGD(learning_rate=hp_lr),
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=["accuracy"])

    return modelo

Y listo, ya hemos creado la función para construir el modelo.

Esta función será llamada por el afinador (*tuner*) y cada vez que la llame se usará una combinación diferente de hiperparámetros, generando por tanto un modelo diferente cada vez que sea ejecutada.

Verifiquemos que al llamar la función no ocurre ningún error:

In [35]:
# Verificar si la función construye el modelo correctamente
import keras_tuner

crear_modelo(keras_tuner.HyperParameters())

<Sequential name=sequential_1, built=True>

¡Perfecto! Ya tenemos el primer paso. Veamos ahora el segundo paso: cómo crear y usar el afinador para entrenar nuestro modelo.

### 6.2. Paso 2: crear usar el *tuner* para afinar los hiperparámetros

Recordemos que tenemos 27 posibles combinaciones de hiper-parámetros, así que la idea sería entrenar y validar 27 diferentes Redes Neuronales y luego escoger aquella con los hiperparámetros que generen el mejor desempeño.

En nuestro caso es posible (pues el set de datos y el modelo son muy simples) pero en la práctica esto no es viable pues se requerirían demasiadas capacidades computacionales y demasiado tiempo de afinación.

Así que en la afinación de hiper-parámetros existen tres enfoques de búsqueda de los mejores hiperparámetros:

- *RandomSearch*: de todas las posibles combinaciones se escogen aleatoriamente algunas de ellas y se afina el modelo con base en esa selección de combinaciones
- *BayesianOptimization*: se usan resultados de combinaciones anteriores para buscar la siguiente combinación que tendrá mejores probabilidades de mejorar el desempeño
- *Hyperband*: se entrenan múltiples modelos con pocas iteraciones y los de peor desempeño se eliminan para luego afinar los modelos más "prometedores" usando más iteraciones

En este caso usaremos "RandomSearch". Comencemos entonces creando el afinador (*tuner*) para este tipo de búsqueda:

In [36]:
tuner = keras_tuner.RandomSearch(
    hypermodel=crear_modelo,  # La función que creamos anteriormente
    objective="val_accuracy", # La métrica de desempeño a usar en la afinación
    max_trials=10, # Número total de combinaciones a probar
    directory="afinacion_red", # Directorio local donde se almacenarán los resultados de entrenamiento
    project_name="trials", # Subdirectorio donde se almacenarán los resultados
    overwrite = True, # Para sobre-escribir los directorios cada vez que afinemos
)

Al hacer lo anterior podemos ver que se crea el directorio "afinacion_red" en el disco duro de la máquina virtual.

Podemos usar el método `search_space_summary()` para imprimir en pantalla la información general del afinador:

In [37]:
# Imprimir en pantalla el espacio de búsqueda
tuner.search_space_summary()

Search space summary
Default search space size: 4
num_layers (Int)
{'default': None, 'conditions': [], 'min_value': 1, 'max_value': 3, 'step': 1, 'sampling': 'linear'}
units_0 (Int)
{'default': None, 'conditions': [], 'min_value': 20, 'max_value': 60, 'step': 20, 'sampling': 'linear'}
activation (Choice)
{'default': 'relu', 'conditions': [], 'values': ['relu', 'tanh'], 'ordered': False}
learning_rate (Choice)
{'default': 0.1, 'conditions': [], 'values': [0.1, 0.01, 0.001], 'ordered': True}


En el caso anterior vemos que el tamaño del espacio de búsqueda es 4 porque queremos afinar 4 hiperparámetros.

Además se imprime en pantalla el rango de valores que tomará cada hiperparámetro durante la afinación.

Y ahora lo único que nos falta es afinar el modelo, para lo cual usamos el método `search()`.

Este método:
- Creará un modelo a partir de la función `crear_modelo` y usando una combinación aleatoria de hiperparámetros
- Tomará los sets de entrenamiento (`x_train`, `y_train`) y validación (`x_val`, `y_val`), se los presentará al modelo y lo entrenará
- Almacenará el modelo entrenado en la carpeta "afinacion_red" así como su desempeño
- Y repetirá el proceso el número de veces especificado en `max_trials`

Y al final de este proceso tendremos un total de `max_trials` modelos y fácilmente podremos escoger el mejor de todos.

Veamos cómo realizar la afinación usando una sola línea de código!:

In [38]:
# Afinar los hiperparámetros
tuner.search(x_train, y_train,
             epochs=3,
             validation_data=(x_val, y_val))

Trial 10 Complete [00h 00m 17s]
val_accuracy: 0.829443097114563

Best val_accuracy So Far: 0.8386568427085876
Total elapsed time: 00h 02m 23s


Observamos que en cada "trial" se imprime en pantalla:

- El tiempo de ejecución del "trial"
- La exactitud obtenida con ese "trial" para el set de validación
- La mejor exactitud con el set de validación obtenida hasta el momento
- Una pequeña tabla con los valores de los hiperparámetros en este "trial" y los mejores valores encontrados hasta el momento

Para modelos relativamente grandes se sugiere usar la GPU de Google Colab para acelerar la afinación.

Una vez completada la afinación podemos ver que la carpeta "afinacion_red" y la sub-carpeta "trials" contienen los diferentes modelos entrenados así como el resumen de los mismos.

Además podemos imprimir en pantalla un resumen de esta afinación usando el método `results_summary()`:

In [ ]:
# Imprimir resumen de la afinación
-tuner.results_summary()

Results summary
Results in afinacion_red/trials
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 01 summary
Hyperparameters:
num_layers: 3
units_0: 60
activation: tanh
learning_rate: 0.01
units_1: 20
units_2: 60
Score: 0.8386568427085876

Trial 04 summary
Hyperparameters:
num_layers: 3
units_0: 40
activation: relu
learning_rate: 0.1
units_1: 60
units_2: 60
Score: 0.8360974788665771

Trial 00 summary
Hyperparameters:
num_layers: 3
units_0: 20
activation: relu
learning_rate: 0.01
units_1: 20
units_2: 20
Score: 0.8333333134651184

Trial 02 summary
Hyperparameters:
num_layers: 2
units_0: 20
activation: tanh
learning_rate: 0.01
units_1: 20
units_2: 40
Score: 0.8296478390693665

Trial 09 summary
Hyperparameters:
num_layers: 3
units_0: 20
activation: relu
learning_rate: 0.01
units_1: 20
units_2: 40
Score: 0.829443097114563

Trial 08 summary
Hyperparameters:
num_layers: 2
units_0: 40
activation: relu
learning_rate: 0.01
units_1: 60
units_2: 60
Score: 0.825143337249

Estos resultados aparecen organizados de mayor a menor, es decir del modelo más exitoso (con mejor desempeño) al menos exitoso (con el peor desempeño).

Por cada uno de los modelos se imprimen los hiperparámetros correspondientes así como el desempeño obtenido.

Y finalmente, sólo nos resta escoger el mejor modelo usando el método `get_best_models`:

In [40]:
# Extraer el mejor modelo
mejor_modelo = tuner.get_best_models(num_models=1)[0]
mejor_modelo.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 60)             │         4,740 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 20)             │         1,220 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 60)             │         1,260 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            61 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,281 (28.44 KB)

 Trainable params: 7,281 (28.44 KB)

 Non-trainable params: 0 (0.00 B)

## 7. Generar predicciones con el modelo afinado

Este es el paso más sencillo de todos, simplemente usamos el modelo que acabamos de seleccionar (`mejor_modelo`) y el método `predict` para generar predicciones sobre nuevos datos.

En este caso recordemos que los datos deben estar pre-procesados y que esta fase ya la implementamos antes.

Por tanto podemos usar `x_test` para generar las predicciones con el mejor modelo:

In [41]:
# Predicciones sobre el set de prueba
logits = mejor_modelo.predict(x_test)
preds = (logits > 0.5).astype(int)
print(preds)

153/153 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
[[0]
 [0]
 [0]
 ...
 [1]
 [0]
 [1]]


¡Y listo! Ya hemos visto cómo afinar una Red Neuronal usando Keras Tuner y como usar el modelo afinado para generar predicciones sobre nuevos datos.